# How an Inference Engine Works?

Infernce Engineering and System Design (Maven Class 5)

Author: Abi Aryan

**Arc:** observe → build (section 3) → break (section 4) → debug (section 5) → towards production

## Section 1. Let's first do the setup

In [78]:
import inspect
import subprocess
import sys
from pathlib import Path

SEED = 0

_shadow = Path.cwd() / "smol_vllm"
if _shadow.is_dir():
    sys.path = [p for p in sys.path if Path(p).resolve() != _shadow.resolve()]
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# After sync: reinstall editable package into *this* kernel's Python
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "./smol-vllm"],
    check=True,
)

for name in list(sys.modules):
    if (
        name == "smol_vllm"
        or name.startswith("smol_vllm.")
        or name.startswith("lib.")
        or name == "agent_demo"
    ):
        del sys.modules[name]

from smol_vllm import LLMEngine, SequenceStatus
from smol_vllm.demo import (
    run_block_manager_checkpoint,
    run_build_allocate_checkpoint,
    run_build_append_checkpoint,
    run_exp1_continuous_batching,
    run_exp2_memory_pressure,
    run_exp3_prefix_sharing,
    run_exp4_throughput_scaling,
    run_exp5_fake_vs_real,
    run_scheduler_checkpoint,
    reproduce_preemption_livelock,
    measure_kv_bytes_per_token,
)
from smol_vllm import exercises
from lib.engine_state import show_engine_state

_params = inspect.signature(LLMEngine.__init__).parameters
assert "seed" in _params, (
    "Old smol_vllm still loaded — use Kernel → Restart, then re-run this cell"
)
print("smol_vllm imported OK (seed, preempt_guard, output_mode, timing available)")

smol_vllm imported OK (seed, preempt_guard, output_mode, timing available)


In [53]:
from lib.gpu_check import load_dotenv, print_gpu_report

load_dotenv()
GPU_REPORT = print_gpu_report()
GPU_OK = GPU_REPORT["real_model_ready"]


Class 5 GPU / Lambda readiness
  smol_vllm import     OK
  torch                OK
  CUDA available       OK
  CUDA device          NVIDIA A100-SXM4-40GB
  nvidia-smi GPU       NVIDIA A100-SXM4-40GB
  transformers         OK
  HF_TOKEN in env      OK
  LAMBDA in .env       OK  (Mac rsync/tunnel)
  Real model ready     OK

  → Part D (GPU) cells can run. First load takes ~30s (TinyLlama).


---
## For a single request ![Engine workflow diagram](assets/engine_workflow.png)

Follow the workflow diagram: request → waiting → allocate → prefill → decode → free.


In [54]:
engine = LLMEngine(
    num_gpu_blocks=24,
    block_size=16,
    max_batch_size=4,
    enable_metrics=True,
    seed=SEED,
)
print(f"Engine: {engine.block_manager.num_blocks} blocks × {engine.block_manager.block_size} tokens/block")
show_engine_state(engine)


Engine: 24 blocks × 16 tokens/block
waiting=[]  running=[]  swapped=[]  | free=24/24 blocks  util=0.0%


In [55]:
# Warm-up: allocate + free so later tables get scattered physical IDs (not [0],[1],[2]...)
for sid, ntok in [(90, 32), (91, 48)]:
    engine.add_request(list(range(ntok)), max_tokens=1)
while engine.scheduler.waiting or engine.scheduler.running:
    engine.step()
# Re-create clean engine for the walkthrough
engine = LLMEngine(num_gpu_blocks=24, block_size=16, max_batch_size=4, enable_metrics=True, seed=SEED)

engine.add_request(list(range(10, 25)), max_tokens=12)
engine.add_request(list(range(100, 130)), max_tokens=8)
engine.add_request(list(range(200, 212)), max_tokens=20)
show_engine_state(engine)

  [metrics] step=1 prefill=8ms decode=0ms gen_tokens=2 tok/s=247 running=0 waiting=0 swapped=0 kv=[░░░░░░░░░░] 0% cpu=0%
waiting=[0, 1, 2]  running=[]  swapped=[]  | free=24/24 blocks  util=0.0%


In [56]:
for gid in [0, 1, 2]:
    ntok = engine.groups[gid].sequences[0].num_tokens
    ok = engine.block_manager.can_allocate(ntok)
    if ok:
        engine.block_manager.allocate(gid, ntok)
    print(f"group {gid}: can_allocate={ok}  block_table={engine.block_manager.get_block_table(gid)}")
show_engine_state(engine)


group 0: can_allocate=True  block_table=[0]
group 1: can_allocate=True  block_table=[1, 2]
group 2: can_allocate=True  block_table=[3]
waiting=[0, 1, 2]  running=[]  swapped=[]  | free=20/24 blocks  util=16.7%


In [57]:
for step in range(1, 8):
    outputs = engine.step()
    show_engine_state(engine, step=step)
    if outputs and all(o.finished for o in outputs):
        break


  [metrics] step=1 prefill=6ms decode=0ms gen_tokens=3 tok/s=515 running=3 waiting=0 swapped=0 kv=[███░░░░░░░] 33% cpu=4%
step  1 | waiting=[]  running=[0, 1, 2]  swapped=[]  | free=16/24 blocks  util=33.3%
  [metrics] step=2 prefill=0ms decode=15ms gen_tokens=3 tok/s=198 running=3 waiting=0 swapped=0 kv=[███░░░░░░░] 38% cpu=0%
step  2 | waiting=[]  running=[0, 1, 2]  swapped=[]  | free=15/24 blocks  util=37.5%
  [metrics] step=3 prefill=0ms decode=15ms gen_tokens=3 tok/s=199 running=3 waiting=0 swapped=0 kv=[████░░░░░░] 42% cpu=0%
step  3 | waiting=[]  running=[0, 1, 2]  swapped=[]  | free=14/24 blocks  util=41.7%
  [metrics] step=4 prefill=0ms decode=15ms gen_tokens=3 tok/s=199 running=3 waiting=0 swapped=0 kv=[████░░░░░░] 42% cpu=0%
step  4 | waiting=[]  running=[0, 1, 2]  swapped=[]  | free=14/24 blocks  util=41.7%
  [metrics] step=5 prefill=0ms decode=15ms gen_tokens=3 tok/s=199 running=3 waiting=0 swapped=0 kv=[████░░░░░░] 46% cpu=0%
step  5 | waiting=[]  running=[0, 1, 2]  swapp

In [58]:

clean = LLMEngine(num_gpu_blocks=24, block_size=16, max_batch_size=4, enable_metrics=True, seed=SEED)
for tok in clean.generate(list(range(5, 25)), max_tokens=8):
    pass
show_engine_state(clean)
clean.metrics.print_summary()


  [metrics] step=1 prefill=2ms decode=0ms gen_tokens=1 tok/s=481 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 8% cpu=0%
  [metrics] step=2 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 8% cpu=0%
  [metrics] step=3 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 8% cpu=0%
  [metrics] step=4 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 8% cpu=0%
  [metrics] step=5 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 8% cpu=0%
  [metrics] step=6 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 8% cpu=0%
  [metrics] step=7 prefill=0ms decode=11ms gen_tokens=1 tok/s=88 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 8% cpu=6%
  [metrics] step=8 prefill=0ms decode=7ms gen_tokens=1 tok/s=141 running=0 waiting=0 swapped=0 kv=[░░░░░░░░░░] 0% cpu=4%
waiting=[]  running=[]  swapped=

---
## 2 — Build it

In [79]:
# 2.1 allocate() — run first. FAIL until you implement exercises.allocate.

# from smol_vllm.reference.exercises_ref import allocate as ref_allocate
# run_build_allocate_checkpoint(ref_allocate)

In [80]:
# 2.2 append_slot() — same pattern

# from smol_vllm.reference.exercises_ref import append_slot as ref_append_slot
# run_build_append_checkpoint(ref_append_slot)

In [61]:
# 2.3 Built-in checkpoints (always run after build section)
run_block_manager_checkpoint()
run_scheduler_checkpoint(seed=SEED)



Checkpoint: BlockSpaceManager
  Allocated 3 seqs × 3 blocks: util=45%
  After appends (16 each): util=60%
  After free(seq 1): util=40%
  Free blocks: 12 (4 blocks returned from seq 1)
  PASS: BlockSpaceManager checkpoint


Checkpoint: Scheduler (continuous batching)
  Submitting 10 requests, max_batch_size=4
  [metrics] step=1 prefill=4ms decode=0ms gen_tokens=4 tok/s=973 running=4 waiting=6 swapped=0 kv=[░░░░░░░░░░] 6% cpu=5%
  step 1: running=4, waiting=6
  [metrics] step=2 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=6 swapped=0 kv=[░░░░░░░░░░] 6% cpu=0%
  step 2: running=4, waiting=6
  [metrics] step=3 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=6 swapped=0 kv=[░░░░░░░░░░] 6% cpu=0%
  step 3: running=4, waiting=6
  [metrics] step=4 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=6 swapped=0 kv=[░░░░░░░░░░] 6% cpu=0%
  step 4: running=4, waiting=6
  [metrics] step=5 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=0 w

---
## 3 — Experiments (predict → run)

Scan the logs for `[preempt]`, `swapped=`, and `running=` — there will be many lines; that's intentional.


### Predict — Exp 1 (continuous batching)

20 requests, `max_batch_size=4`.


In [62]:
exp1_engine = run_exp1_continuous_batching(seed=SEED)


Experiment 1: Continuous Batching
Submit 20 requests of different lengths
Columns: step | running | waiting | swapped | blocks_used (%)

  [metrics] step=1 prefill=18ms decode=0ms gen_tokens=4 tok/s=221 running=4 waiting=16 swapped=0 kv=[█░░░░░░░░░] 19% cpu=4%
  step   1 | running= 4 | waiting=16 | swapped= 0 | blocks_used= 18.8%
  [metrics] step=2 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=16 swapped=0 kv=[██░░░░░░░░] 22% cpu=0%
  step   2 | running= 4 | waiting=16 | swapped= 0 | blocks_used= 21.9%
  [metrics] step=3 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=16 swapped=0 kv=[██░░░░░░░░] 22% cpu=0%
  step   3 | running= 4 | waiting=16 | swapped= 0 | blocks_used= 21.9%
  [metrics] step=4 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=16 swapped=0 kv=[██░░░░░░░░] 22% cpu=2%
  step   4 | running= 4 | waiting=16 | swapped= 0 | blocks_used= 21.9%
  [metrics] step=5 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waitin

### Predict — Exp 2 (memory pressure)

`num_blocks=16`, `block_size=16`


In [63]:
exp2_engine = run_exp2_memory_pressure(seed=SEED)


Experiment 2: Memory Pressure & Preemption
num_blocks=16, 10 long sequences (64 tokens each) -> watch preemption

  [metrics] step=1 prefill=19ms decode=0ms gen_tokens=3 tok/s=155 running=3 waiting=7 swapped=0 kv=[█████████░] 94% cpu=1%
  step   1 | running= 3 | swapped= 0 | util=94%
  [preempt] group 2 -> swapped (blocks freed)
  [metrics] step=2 prefill=0ms decode=10ms gen_tokens=2 tok/s=198 running=2 waiting=7 swapped=1 kv=[██████░░░░] 62% cpu=0%
  step   2 | running= 2 | swapped= 1 | util=62%
  [metrics] step=3 prefill=0ms decode=10ms gen_tokens=2 tok/s=198 running=2 waiting=7 swapped=1 kv=[██████░░░░] 62% cpu=0%
  step   3 | running= 2 | swapped= 1 | util=62%
  [metrics] step=4 prefill=0ms decode=10ms gen_tokens=2 tok/s=198 running=2 waiting=7 swapped=1 kv=[██████░░░░] 62% cpu=0%
  step   4 | running= 2 | swapped= 1 | util=62%
  [metrics] step=5 prefill=0ms decode=10ms gen_tokens=2 tok/s=198 running=2 waiting=7 swapped=1 kv=[██████░░░░] 62% cpu=3%
  step   5 | running= 2 | swappe

### Predict — Exp 3 (prefix sharing)

5 seqs, 32-token shared prefix, `block_size=16`


In [64]:
run_exp3_prefix_sharing()


Experiment 3: Prefix Sharing (copy_on_write)
5 sequences with same 32-token prefix -> compare utilization

  Without sharing — block tables:
    seq 0: [0, 1, 2]
    seq 1: [3, 4, 5]
    seq 2: [6, 7, 8]
    seq 3: [9, 10, 11]
    seq 4: [12, 13, 14]
  ref_counts: {0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1}

  Without prefix sharing: 5 seqs × 3 blocks = 15 blocks -> util=23%
  With copy_on_write: shared prefix -> util=11%

  With sharing — block tables (note shared block IDs):
    seq 0: [15, 16, 17]
    seq 1: [15, 16, 17, 18]
    seq 2: [15, 16, 17, 19]
    seq 3: [15, 16, 17, 20]
    seq 4: [15, 16, 17, 21]
  ref_counts: {15: 5, 16: 5, 17: 5, 18: 1, 19: 1, 20: 1, 21: 1}
  Shared prefix block IDs: [15, 16]



---
## 4 — Break it

In [65]:
# 4.1 Block pool: 8 blocks, 64-token prompts need 4 blocks each → 2 fit, 3rd waits
from smol_vllm import BlockSpaceManager

bm = BlockSpaceManager(num_blocks=8, block_size=16)
for sid in [0, 1]:
    bm.allocate(sid, 64)
    print(f"seq {sid}: table={bm.get_block_table(sid)}")
print(f"After 2 seqs: free={bm.num_free_blocks()}/8  (3rd seq would need 4 — queue or reject)")

print("\nScheduler admission (free_after >= running rule):")
break1 = LLMEngine(num_gpu_blocks=8, block_size=16, max_batch_size=8, seed=SEED)
for i in range(3):
    break1.add_request(list(range(1000 + i, 1064 + i)), max_tokens=5)
for step in range(8):
    break1.step()
    show_engine_state(break1, step=step)


seq 0: table=[0, 1, 2, 3]
seq 1: table=[4, 5, 6, 7]
After 2 seqs: free=0/8  (3rd seq would need 4 — queue or reject)

Scheduler admission (free_after >= running rule):
  [metrics] step=1 prefill=6ms decode=0ms gen_tokens=1 tok/s=154 running=1 waiting=2 swapped=0 kv=[██████░░░░] 62% cpu=5%
step  0 | waiting=[1, 2]  running=[0]  swapped=[]  | free=3/8 blocks  util=62.5%
  [metrics] step=2 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=2 swapped=0 kv=[██████░░░░] 62% cpu=0%
step  1 | waiting=[1, 2]  running=[0]  swapped=[]  | free=3/8 blocks  util=62.5%
  [metrics] step=3 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=2 swapped=0 kv=[██████░░░░] 62% cpu=0%
step  2 | waiting=[1, 2]  running=[0]  swapped=[]  | free=3/8 blocks  util=62.5%
  [metrics] step=4 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=2 swapped=0 kv=[██████░░░░] 62% cpu=0%
step  3 | waiting=[1, 2]  running=[0]  swapped=[]  | free=3/8 blocks  util=62.5%
  [metrics] step=5 p

In [66]:
# 4.2 block_size trade-off — hold ~128-token capacity, same 64-token prompt
import math

PROMPT_TOKENS = 64
CAPACITY_TOKENS = 128

for bs in [1, 16, 64]:
    num_blocks = math.ceil(CAPACITY_TOKENS / bs)
    e = LLMEngine(num_gpu_blocks=num_blocks, block_size=bs, max_batch_size=1, seed=SEED)
    e.add_request(list(range(PROMPT_TOKENS)), max_tokens=4)
    e.step()
    table = e.block_manager.get_block_table(0)
    blocks_used = len(table)
    slots = blocks_used * bs
    wasted = slots - PROMPT_TOKENS
    print(
        f"block_size={bs:2d}  blocks={num_blocks:3d}  table_len={blocks_used:2d}  "
        f"wasted_slots={wasted:2d}  table={table}"
    )


  [metrics] step=1 prefill=6ms decode=0ms gen_tokens=1 tok/s=154 running=1 waiting=0 swapped=0 kv=[█████░░░░░] 51% cpu=2%
block_size= 1  blocks=128  table_len=65  wasted_slots= 1  table=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64]
  [metrics] step=1 prefill=6ms decode=0ms gen_tokens=1 tok/s=154 running=1 waiting=0 swapped=0 kv=[██████░░░░] 62% cpu=4%
block_size=16  blocks=  8  table_len= 5  wasted_slots=16  table=[0, 1, 2, 3, 4]
  [metrics] step=1 prefill=6ms decode=0ms gen_tokens=1 tok/s=154 running=1 waiting=0 swapped=0 kv=[██████████] 100% cpu=0%
block_size=64  blocks=  2  table_len= 2  wasted_slots=64  table=[0, 1]


In [67]:
# 4.3 max_batch_size=64 but num_gpu_blocks=24 — which limit binds?
break3 = LLMEngine(num_gpu_blocks=24, block_size=16, max_batch_size=64, seed=SEED)
for i in range(12):
    break3.add_request(list(range(50)), max_tokens=10)
for step in range(6):
    break3.step()
    print(f"step {step+1}: running={len(break3.scheduler.running)} waiting={len(break3.scheduler.waiting)}")


  [metrics] step=1 prefill=20ms decode=0ms gen_tokens=4 tok/s=199 running=4 waiting=8 swapped=0 kv=[██████░░░░] 67% cpu=1%
step 1: running=4 waiting=8
  [metrics] step=2 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=8 swapped=0 kv=[██████░░░░] 67% cpu=0%
step 2: running=4 waiting=8
  [metrics] step=3 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=8 swapped=0 kv=[██████░░░░] 67% cpu=0%
step 3: running=4 waiting=8
  [metrics] step=4 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=8 swapped=0 kv=[██████░░░░] 67% cpu=0%
step 4: running=4 waiting=8
  [metrics] step=5 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=8 swapped=0 kv=[██████░░░░] 67% cpu=6%
step 5: running=4 waiting=8
  [metrics] step=6 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=8 swapped=0 kv=[██████░░░░] 67% cpu=0%
step 6: running=4 waiting=8


---
## 5 — Debug it

In [68]:
reproduce_preemption_livelock(seed=SEED)


Debug: preemption livelock (preempt_guard=False)
  [metrics] step=1 prefill=40ms decode=0ms gen_tokens=1 tok/s=25 running=1 waiting=0 swapped=0 kv=[█████████░] 92% cpu=2%
  [metrics] step=2 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[█████████░] 92% cpu=0%
  [metrics] step=3 prefill=0ms decode=5ms gen_tokens=1 tok/s=197 running=1 waiting=0 swapped=0 kv=[█████████░] 96% cpu=0%
  [preempt] group 0 -> swapped (blocks freed)
  [swap-in] group 0 <- swapped
  [metrics] step=4 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[█████████░] 96% cpu=0%
  [preempt] group 0 -> swapped (blocks freed)
  [swap-in] group 0 <- swapped
  [metrics] step=5 prefill=0ms decode=5ms gen_tokens=1 tok/s=195 running=1 waiting=0 swapped=0 kv=[█████████░] 96% cpu=0%
  [preempt] group 0 -> swapped (blocks freed)
  [swap-in] group 0 <- swapped
  [metrics] step=6 prefill=0ms decode=5ms gen_tokens=1 tok/s=197 running=1 waiting=0 swapped=0 kv=[█████████░

In [69]:
# Fixed — default preempt_guard=True
fixed = LLMEngine(num_gpu_blocks=24, block_size=16, max_batch_size=4, seed=SEED, preempt_guard=True)
fixed.add_request(list(range(350)), max_tokens=12)
for step in range(20):
    fixed.step()
show_engine_state(fixed)


  [metrics] step=1 prefill=35ms decode=0ms gen_tokens=1 tok/s=28 running=1 waiting=0 swapped=0 kv=[█████████░] 92% cpu=1%
  [metrics] step=2 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[█████████░] 92% cpu=0%
  [metrics] step=3 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[█████████░] 96% cpu=0%
  [metrics] step=4 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[█████████░] 96% cpu=0%
  [metrics] step=5 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[█████████░] 96% cpu=0%
  [metrics] step=6 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[█████████░] 96% cpu=0%
  [metrics] step=7 prefill=0ms decode=5ms gen_tokens=1 tok/s=197 running=1 waiting=0 swapped=0 kv=[█████████░] 96% cpu=0%
  [metrics] step=8 prefill=0ms decode=7ms gen_tokens=1 tok/s=150 running=1 waiting=0 swapped=0 kv=[█████████░] 96% cpu=9%
  [metrics] step=9 prefi

---
## 6 — Throughput & simulator (Exp 4)

In [70]:
naive_engines = run_exp4_throughput_scaling(timing="naive", seed=SEED)
roof_engines = run_exp4_throughput_scaling(timing="roofline", seed=SEED)



Experiment 4: Throughput Scaling (timing=naive)
  [naive] cost ∝ batch_size only — sleep(0.005 × batch_size) per decode step (no fixed overhead)
Measure tok/s at batch_size=1, 8, 16 -> ASCII bar chart
(step metrics suppressed here so the bar chart stays visible)

  batch_size= 1: 195.6 tok/s
  batch_size= 8: 199.1 tok/s
  [preempt] group 11 -> swapped (blocks freed)
  [preempt] group 10 -> swapped (blocks freed)
  batch_size=16: 199.1 tok/s

  Throughput (tok/s) bar chart:
    batch= 1 |███████████████████████████████████████░| 195.6
    batch= 8 |███████████████████████████████████████░| 199.1
    batch=16 |████████████████████████████████████████| 199.1

  [metrics] summary:
    prompt_tokens_total=0 generation_tokens_total=0
    time_to_first_token_avg_ms=0 tpot_avg_ms=0 e2e_request_latency_avg_ms=0
    prompt_len_avg=0 (prefix tokens at start)



Experiment 4: Throughput Scaling (timing=roofline)
  [roofline] fixed + marginal — sleep(0.02 + 0.0005 × batch_size) per decode step
Mea

---
## 7 — On GPU (Lambda / CUDA)

Skip if `GPU_OK` is false. Derive slide numbers from your own `[metrics]` lines.


In [71]:
if GPU_OK:
    real_engine = run_exp5_fake_vs_real(seed=SEED)
else:
    print("GPU track skipped — see LAMBDA.md")



Experiment 5: Fake vs Real Model (Educational)

=== Fake model (simulated timing, zero deps) ===
  [metrics] step=1 prefill=8ms decode=0ms gen_tokens=4 tok/s=494 running=4 waiting=0 swapped=0 kv=[░░░░░░░░░░] 6% cpu=0%
  [metrics] step=2 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=0 swapped=0 kv=[░░░░░░░░░░] 6% cpu=0%
  [metrics] step=3 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=0 swapped=0 kv=[░░░░░░░░░░] 6% cpu=0%
  [metrics] step=4 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=0 swapped=0 kv=[░░░░░░░░░░] 6% cpu=0%
  [metrics] step=5 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=0 swapped=0 kv=[░░░░░░░░░░] 6% cpu=0%
  [metrics] step=6 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=0 swapped=0 kv=[░░░░░░░░░░] 6% cpu=0%
  [metrics] step=7 prefill=0ms decode=20ms gen_tokens=4 tok/s=199 running=4 waiting=0 swapped=0 kv=[░░░░░░░░░░] 6% cpu=0%
  [metrics] step=8 prefill=0ms decode=20ms gen_to

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[smol-vllm] Loaded in 1.2s
[edu] Prefill batch=2 compute-bound prompt_tokens≈6 26ms → [13, 2]
  [metrics] step=1 prefill=28ms decode=0ms gen_tokens=2 tok/s=72 running=2 waiting=0 swapped=0 kv=[░░░░░░░░░░] 2% gpu_mem=8417/8456MB cpu=9%
[edu] Decode batch=2 memory-bound KV cache reads 42ms → [13, 29871]
  [metrics] step=2 prefill=0ms decode=42ms gen_tokens=2 tok/s=47 running=2 waiting=0 swapped=0 kv=[░░░░░░░░░░] 2% gpu_mem=8417/8456MB cpu=3%
[edu] Decode batch=2 memory-bound KV cache reads 40ms → [29961, 13]
  [metrics] step=3 prefill=0ms decode=40ms gen_tokens=2 tok/s=50 running=2 waiting=0 swapped=0 kv=[░░░░░░░░░░] 2% gpu_mem=8417/8456MB cpu=3%
[edu] Decode batch=2 memory-bound KV cache reads 41ms → [10858, 29966]
  [metrics] step=4 prefill=0ms decode=41ms gen_tokens=2 tok/s=49 running=2 waiting=0 swapped=0 kv=[░░░░░░░░░░] 2% gpu_mem=8418/8456MB cpu=3%
[edu] Decode batch=2 memory-bound KV cache reads 40ms → [4408, 29989]
  [metrics] step=5 prefill=0ms decode=40ms gen_tokens=2 tok/s=50 

In [72]:
# Prefill vs decode asymmetry + KV bytes/token (when GPU_OK)
if GPU_OK:
    m = real_engine.metrics
    if m.prefill_latencies and m.decode_latencies:
        p = sum(m.prefill_latencies)/len(m.prefill_latencies)*1000
        d = sum(m.decode_latencies)/len(m.decode_latencies)*1000
        print(f"prefill_avg={p:.0f}ms  decode_avg={d:.0f}ms  ratio={p/max(d,0.001):.1f}x")
    try:
        import torch
        if torch.cuda.is_available():
            mb = torch.cuda.max_memory_allocated()/1024**2
            print(f"VRAM peak={mb:.0f} MB  (TinyLlama ~1.1B × 2B ≈ 2.2 GB weights)")
    except ImportError:
        pass
    measure_kv_bytes_per_token(seed=SEED)


prefill_avg=28ms  decode_avg=41ms  ratio=0.7x
VRAM peak=8419 MB  (TinyLlama ~1.1B × 2B ≈ 2.2 GB weights)
[smol-vllm] Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 on cuda ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[smol-vllm] Loaded in 1.3s
[edu] Prefill batch=1 compute-bound prompt_tokens≈18 22ms → [13]
  [metrics] step=1 prefill=23ms decode=0ms gen_tokens=1 tok/s=44 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 2% gpu_mem=8417/8456MB cpu=8%
[edu] Decode batch=1 memory-bound KV cache reads 20ms → [13]
  [metrics] step=2 prefill=0ms decode=20ms gen_tokens=1 tok/s=49 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 2% gpu_mem=8417/8456MB cpu=3%
[edu] Decode batch=1 memory-bound KV cache reads 19ms → [29906]
  [metrics] step=3 prefill=0ms decode=19ms gen_tokens=1 tok/s=51 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 2% gpu_mem=8418/8456MB cpu=2%
[edu] Decode batch=1 memory-bound KV cache reads 20ms → [29889]
  [metrics] step=4 prefill=0ms decode=20ms gen_tokens=1 tok/s=51 running=1 waiting=0 swapped=0 kv=[░░░░░░░░░░] 2% gpu_mem=8418/8456MB cpu=3%
[edu] Decode batch=1 memory-bound KV cache reads 19ms → [4803]
  [metrics] step=5 prefill=0ms decode=20ms gen_tokens=1 tok/s=51 running=1 waiting=0 swapped

---
## 8 — Real traffic — CrewAI through your engine


### Two layers — same stack as production

| Layer | Job | In this notebook | What you see in logs |
|-------|-----|------------------|----------------------|
| **Orchestration** (CrewAI) | Break a user task into steps; route work between agents; pass context task → task | Research Analyst → Technical Writer | CrewAI task/agent banners (green/purple UI) |
| **Inference** (smol_vllm) | Schedule requests; allocate KV blocks; run prefill/decode; return tokens | **One shared** `LLMEngine` for the whole crew | `ENGINE REQUEST #1`, `#2`, `waiting` / `running` / `swapped`, `[metrics]` |
| **Model** | Turn tokens into next tokens | FakeModel `output_mode="text"` on CPU (optional TinyLlama on GPU) | Readable fragment vs real answer |

In [73]:
from agent_demo import build_crew, kickoff_crew

agent_engine = LLMEngine(
    num_gpu_blocks=24,
    block_size=16,
    max_batch_size=4,
    enable_metrics=True,
    seed=SEED,
    output_mode="text",
)
crew = build_crew(agent_engine, verbose_engine=True, dual_agent=True)
user_query = "What is paged attention in LLM inference?"
print(f"USER: {user_query}")
print("\nExpect: ENGINE REQUEST #1 (Research Analyst) → #2 (Technical Writer)")


USER: What is paged attention in LLM inference?

Expect: ENGINE REQUEST #1 (Research Analyst) → #2 (Technical Writer)


In [74]:
result = await kickoff_crew(crew, {"user_query": user_query})
print(result.raw)
agent_engine.metrics.print_summary()


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 652e9702-6c1a-4db5-a0e0-cba689a3a7bb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: The user asked: What is paged attention in LLM inference?                                                │
│                                                                                                                 │
│  Use the inference engine to list 3 key facts about this topic.                                                 │
│  ID: 6e09369f-b04f-4612-b5cb-0f5299d0131d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Task: The user asked: What is paged attention in LLM inference?                                                │
│                                                                                                                 │
│  Use the inference engine to list 3 key facts about this topic.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


ENGINE REQUEST #1 — CrewAI agent: Research Analyst
[engine] prompt truncated to 348 tokens (KV block budget)
prompt (348 tokens): "system: You are Research Analyst. You gather bullet-point facts. Every LLM call goes through smol_vllm.\nYour personal goal is: Extract key facts about the user's topic via the inference engine.\nuser: ..."

[engine] request queued — scheduler state:
waiting=[0]  running=[]  swapped=[]  | free=24/24 blocks  util=0.0%


  [metrics] step=1 prefill=35ms decode=0ms gen_tokens=1 tok/s=29 running=1 waiting=0 swapped=0 kv=[█████████░] 92% cpu=0%
step  1 | waiting=[]  running=[0]  swapped=[]  | free=2/24 blocks  util=91.7%
  [metrics] step=2 prefill=0ms decode=5ms gen_tokens=1 tok/s=196 running=1 waiting=0 swapped=0 kv=[█████████░] 92% cpu=0%
step  2 | waiting=[]  running=[0]  swapped=[]  | free=2/24 blocks  util=91.7%
  [metrics] step=3 prefill=0ms decode=5ms gen_tokens=1 tok/s=197 running=1 waiting=0 swapped=0 kv=[█████████░] 92% cpu=0%
step  3 | waiting=[]  running=[0]  swapped=[]  | free=2/24 blocks  util=91.7%
  [metrics] step=4 prefill=0ms decode=5ms gen_tokens=1 tok/s=197 running=1 waiting=0 swapped=0 kv=[█████████░] 92% cpu=0%
step  4 | waiting=[]  running=[0]  swapped=[]  | free=2/24 blocks  util=91.7%
  [metrics] step=5 prefill=0ms decode=5ms gen_tokens=1 tok/s=197 running=1 waiting=0 swapped=0 kv=[█████████░] 96% cpu=0%
step  5 | waiting=[]  running=[0]  swapped=[]  | free=1/24 blocks  util=95.8%


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ng in large language                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: The user asked: What is paged attention in LLM inference?                                                │
│                                                                                                                 │
│  Using the research notes from the previous task, write a clear 2-sentence answer.                              │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 652e9702-6c1a-4db5-a0e0-cba689a3a7bb                                                                       │
│  Final Output: ng in large language                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ng in large language

  [metrics] summary:
    prompt_tokens_total=696 generation_tokens_total=40
    time_to_first_token_avg_ms=35 tpot_avg_ms=6 e2e_request_latency_avg_ms=144
    prompt_len_avg=348 (prefix tokens at start)
    prefill_latency_avg_ms=35
    decode_latency_avg_ms=5



╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [76]:
# Optional: CrewAI on GPU (when Lambda/CUDA ready)
if GPU_OK:
    gpu_crew_engine = LLMEngine(
        num_gpu_blocks=128,
        block_size=16,
        max_batch_size=2,
        use_real_model=True,
        enable_metrics=True,
        seed=SEED,
    )
    gpu_crew = build_crew(gpu_crew_engine, verbose_engine=True, dual_agent=True)
    gpu_result = await kickoff_crew(gpu_crew, {"user_query": user_query})
    print(gpu_result.raw)
    gpu_crew_engine.metrics.print_summary()


[smol-vllm] Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 on cuda ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[smol-vllm] Loaded in 1.2s


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 808e6e0c-af72-4c37-8233-9f5801fa17e7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: The user asked: What is paged attention in LLM inference?                                                │
│                                                                                                                 │
│  Use the inference engine to list 3 key facts about this topic.                                                 │
│  ID: 6326b72d-6d91-4e03-98e8-c59ff03dd5bf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Paged attention is a type of attention that is focused on a specific topic or aspect of                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: The user asked: What is paged attention in LLM inference?                                                │
│                                                                                                                 │
│  Using the research notes from the previous task, write a clear 2-sentence answer.                              │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 808e6e0c-af72-4c37-8233-9f5801fa17e7                                                                       │
│  Final Output:                                                                                                  │
│                                                                                                                 │
│  Paged attention is a type of attention that is focused on a specific topic or aspect of                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



Paged attention is a type of attention that is focused on a specific topic or aspect of

  [metrics] summary:
    prompt_tokens_total=306 generation_tokens_total=40
    time_to_first_token_avg_ms=33 tpot_avg_ms=20 e2e_request_latency_avg_ms=421
    prompt_len_avg=153 (prefix tokens at start)
    prefill_latency_avg_ms=32
    decode_latency_avg_ms=20



╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 9 — Diff to production vLLM

Open side by side:

| smol_vllm | production |
|-----------|------------|
| `scheduler.py` → `schedule()` | `vllm/v1/core/sched/scheduler.py` |
| waiting / running / swapped | same skeleton |
| allocate + preempt | + chunked prefill, prefix cache, speculative decode, grammar bitmasks |

The toy wasn't a toy. It's the control plane with the extras removed.


---
## Export PDF (optional)

Run after all cells execute.


In [77]:
from lib.export_notebook import save_pdf
from IPython.display import FileLink

pdf = save_pdf("class5.ipynb")
FileLink(pdf.name)


/home/ubuntu/class5/class5.ipynb.pdf